# Evaluate indication mapping

This notebook isolates one question: can a single LLM call map existing MOAlmanac indications to indications from the latest label and correctly classify each relationship as `same`, `revised`, `new`, `not_found`, or `uncertain`?

Proposal generation, changelog assessment, and approval-date matching are intentionally out of scope here.

In [1]:
import json
import os
from collections import Counter
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

from moalmanac_fda_curation.core.extract_indications_from_fda_label import (
    DEFAULT_MAX_TOKENS,
    DEFAULT_MODEL,
    extract_indications_from_label_url,
)
from moalmanac_fda_curation.core.reconcile_indications import (
    build_reconciliation_prompt,
    indexed_latest_indications,
    load_existing_indications,
    reconcile_indications,
)

## Configure the Opdivo comparison

The existing records come from MOAlmanac. The local Opdivo changelog is used only to identify the newest label URL; its events are not passed to the mapping LLM.

In [2]:
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("No .env file found")
load_dotenv(env_path)
if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(f"ANTHROPIC_API_KEY is missing from {env_path}")

PROJECT_ROOT = Path(env_path).parent
WORKSPACE_ROOT = PROJECT_ROOT.parent
EXISTING_INDICATIONS_JSON = WORKSPACE_ROOT / "moalmanac-db/referenced/indications.json"
CHANGELOG_JSON = WORKSPACE_ROOT / "ai-assisted-gk-curation/analyses/fda-indications/extracted-indications/section1-changelogs/Opdivo-bla125554-section1-changelog.json"
EXTRACTION_DIR = PROJECT_ROOT / "analyses/reconciliation-inputs/opdivo-current-label"

changelog = json.loads(CHANGELOG_JSON.read_text())
latest_event = max(
    changelog["events"],
    key=lambda event: (event["date"], event["event_number"]),
)
LATEST_LABEL_URL = latest_event["label_url"]
print("Latest label date:", latest_event["date"])
print("Latest label URL:", LATEST_LABEL_URL)

Latest label date: 2026-03-20
Latest label URL: http://www.accessdata.fda.gov/drugsatfda_docs/label/2026/125554s135lbl.pdf


## Optionally regenerate latest-label indications

Leave `REGENERATE = False` to evaluate mapping against the saved extraction. Set it to `True` only when you want to download the current label and rerun the indication-extraction LLM call.

In [3]:
REGENERATE = False

if REGENERATE:
    extraction_paths = extract_indications_from_label_url(
        label_url=LATEST_LABEL_URL,
        brand_name="Opdivo",
        generic_name="nivolumab",
        application_number="BLA125554",
        labels_dir=EXTRACTION_DIR / "labels",
        indications_dir=EXTRACTION_DIR / "intermediate",
        model=DEFAULT_MODEL,
        max_tokens=DEFAULT_MAX_TOKENS,
        overwrite=True,
    )
else:
    extraction_paths = {
        "claude_chunked_indication_fields": (
            EXTRACTION_DIR
            / "intermediate/Opdivo-BLA125554-claude_chunked_indication_fields.json"
        )
    }

LATEST_EXTRACTION_JSON = extraction_paths["claude_chunked_indication_fields"]
print("Extraction artifact:", LATEST_EXTRACTION_JSON)

Extraction artifact: /Users/sabrina/Documents/VA/MOAlmanac/moalmanac-fda-curation/analyses/reconciliation-inputs/opdivo-current-label/intermediate/Opdivo-BLA125554-claude_chunked_indication_fields.json


## Load and inspect the indication strings

Only indication IDs, extraction indexes, and indication strings are supplied to the mapping call. `biomarker_only=True` keeps this experiment aligned with the current MOAlmanac scope.

In [4]:
existing_indications = load_existing_indications(
    EXISTING_INDICATIONS_JSON, document_id="doc:fda.opdivo"
)
latest_payload = json.loads(LATEST_EXTRACTION_JSON.read_text())
latest_indications = indexed_latest_indications(latest_payload, biomarker_only=True)

print(f"Existing indications: {len(existing_indications)}")
for indication in existing_indications:
    print(f"  {indication['id']}: {indication['indication']}")

print(f"\nLatest-label indications: {len(latest_indications)}")
for indication in latest_indications:
    print(
        f"  [{indication['latest_indication_index']}]: "
        f"{indication['indication']}"
    )

Existing indications: 5
  ind:fda.opdivo:0: OPDIVO is a programmed death receptor-1 (PD-1) blocking antibody indicated for the treatment of adult patients with resectable (tumors >=4 cm or node positive) non-small cell lung cancer and no known EGFR mutations or ALK rearrangements, for neoadjuvant treatment, in combination with platinum-doublet chemotherapy, followed by single-agent OPDIVO as adjuvant treatment after surgery.
  ind:fda.opdivo:1: OPDIVO is a programmed death receptor-1 (PD-1)-blocking antibody indicated for the treatment of adult patients with metastatic non-small cell lung cancer expressing PD-L1 (>= 1%) as determined by an FDA-approved test, with no EGFR or ALK genomic tumor aberrations, as first-line treatment in combination with ipilimumab.
  ind:fda.opdivo:2: OPDIVO is a programmed death receptor-1 (PD-1)-blocking antibody indicated for the treatment of adult patients with metastatic or recurrent non-small cell lung cancer with no EGFR or ALK genomic tumor aberratio

## Inspect the exact prompt

Read this before running the LLM. It is the complete mapping prompt, including both indication lists and the definitions of `same` and `revised`.

In [5]:
mapping_prompt = build_reconciliation_prompt(
    existing_indications, latest_indications
)
print(mapping_prompt)

# Task

Map existing curated MOAlmanac FDA indications to indications extracted from a
newer FDA label, and classify every resulting relationship.

For each possible pair, make two judgments in order:

1. Do the records represent the same underlying FDA indication?
2. If they do, has the indication been updated?

# Existing MOAlmanac indications

```json
[
  {
    "id": "ind:fda.opdivo:0",
    "indication": "OPDIVO is a programmed death receptor-1 (PD-1) blocking antibody indicated for the treatment of adult patients with resectable (tumors >=4 cm or node positive) non-small cell lung cancer and no known EGFR mutations or ALK rearrangements, for neoadjuvant treatment, in combination with platinum-doublet chemotherapy, followed by single-agent OPDIVO as adjuvant treatment after surgery."
  },
  {
    "id": "ind:fda.opdivo:1",
    "indication": "OPDIVO is a programmed death receptor-1 (PD-1)-blocking antibody indicated for the treatment of adult patients with metastatic non-small cell lu

## Run one mapping call

This is the notebook's only LLM evaluation step. The verification checks that every indication is accounted for exactly once and that every `revised` result includes at least one meaningful wording difference.

In [6]:
reconciliation = reconcile_indications(
    existing_indications, latest_indications
)

print("Verified:", reconciliation["verified"])
print("Verification errors:", reconciliation["verification_errors"])
print(
    "Classification counts:",
    Counter(item["classification"] for item in reconciliation["mappings"]),
)

Verified: True
Verification errors: []
Classification counts: Counter({'same': 3, 'new': 3, 'revised': 2})


## Review every decision

The paired strings are shown together with the model's final classification, rationale, and any meaningful wording differences. This is the main evaluation output.

In [8]:
for number, mapping in enumerate(reconciliation["mappings"], start=1):
    existing = mapping["existing_indication"]
    latest = mapping["latest_indication"]
    print("=" * 100)
    print(f"MAPPING {number}: {mapping['classification'].upper()}")
    print("Existing ID:", mapping["existing_indication_id"])
    print("Latest index:", mapping["latest_indication_index"])
    print("Existing:", existing["indication"] if existing else None)
    print("Latest:  ", latest["indication"] if latest else None)
    print("Reason:  ", mapping["reason"])
    if mapping["differences"]:
        print("Meaningful differences:")
        for difference in mapping["differences"]:
            print("  Existing wording:", difference["existing_wording"])
            print("  Latest wording:  ", difference["latest_wording"])
            print("  Difference:       ", difference["difference"])

MAPPING 1: REVISED
Existing ID: ind:fda.opdivo:0
Latest index: 4
Existing: OPDIVO is a programmed death receptor-1 (PD-1) blocking antibody indicated for the treatment of adult patients with resectable (tumors >=4 cm or node positive) non-small cell lung cancer and no known EGFR mutations or ALK rearrangements, for neoadjuvant treatment, in combination with platinum-doublet chemotherapy, followed by single-agent OPDIVO as adjuvant treatment after surgery.
Latest:   Opdivo is a programmed death receptor-1 (PD-1)-blocking antibody indicated, in combination with platinum-doublet chemotherapy, for the neoadjuvant treatment of adult patients with resectable (tumors ≥4 cm or node positive) NSCLC and no known epidermal growth factor receptor (EGFR) mutations or anaplastic lymphoma kinase (ALK) rearrangements, followed by single-agent Opdivo (nivolumab) as adjuvant treatment after surgery.
Reason:   Same neoadjuvant NSCLC indication with minor wording changes including abbreviation of disease 

## Optionally save this run

In [ ]:
SAVE_RESULTS = False
RESULTS_JSON = EXTRACTION_DIR / "opdivo-indication-mapping.json"

if SAVE_RESULTS:
    RESULTS_JSON.write_text(json.dumps(reconciliation, indent=2) + "\n")
    print("Saved:", RESULTS_JSON)